# Customization triage: prompt vs RAG vs fine-tune, decided on evidence

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 48 §48.1 · type: concept-lab*

*One-line promise:* turn the chapter's three-lever triage into a small **decision tool**, run real scenarios through it, and defend a prompt-vs-RAG-vs-fine-tune call with a *measured* signal instead of fashion.

## 🧠 Why this matters

"Should we fine-tune?" is the most expensive question a team answers on vibes. The three levers change *different things*: **prompting changes instructions**, **RAG changes available knowledge**, **fine-tuning changes the weights — the model's default behavior**. Reach for them top to bottom — prompt first, retrieve second, train only when the first two *demonstrably plateau on a measured eval*. The senior skill isn't knowing the levers; it's matching each task to the cheapest one that moves the metric, and knowing that the eval is the prerequisite, not the afterthought.

## Objectives & prereqs

**By the end you can:**
- Map a task to the right lever (prompt / RAG / fine-tune) and say *what each one changes*.
- Run a set of real scenarios through a reusable decision function and check yourself against gold answers.
- Explain why fine-tuning to *inject knowledge* is the classic failure — and where facts vs form actually belong.
- Use a measured *plateau* as the trigger for training: "an agent you cannot evaluate is an agent you cannot train."

**Prereqs:** Ch 10 (prompting) · Ch 13 (RAG) · Ch 22 (evals — the measured plateau). Run the setup cell first.

**Cost:** none. Fully offline — a decision function plus a mocked metric. No API key, no network.

## Setup

In [ ]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads a git-ignored .env if present; never hardcode keys

# This notebook is 100% offline: a decision function and a mocked metric. The
# MOCK switch is here for consistency with the rest of the course; no live model
# is ever called from 48-01.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(7)  # the mocked plateau demo below is reproducible

DATA = Path("data")


def load_scenarios():
    return json.loads((DATA / "scenarios.json").read_text(encoding="utf-8"))


print("MOCK =", MOCK, "| this notebook runs fully offline (no model calls)")

## The three levers — what each one *changes*

Pin the mental model before the mechanics. Same axis the chapter draws: you **tell** someone the dress code (prompt), you **hand them the manual** (RAG), or you **apprentice them until the habit is automatic** (fine-tune). Habits are powerful and cheap to execute — and slow and costly to retrain when requirements change next quarter.

In [ ]:
LEVERS = {
    "prompt": {
        "changes": "instructions",
        "reach_when": "Behavior, tone, format, or workflow needs to change. Always the "
                      "first move: fast, cheap, reversible, no infra.",
        "wont": "Hold beyond the context limit, or add a capability the model lacks.",
    },
    "rag": {
        "changes": "available knowledge",
        "reach_when": "Answers depend on private, large, or changing knowledge; you need "
                      "citations and instant updates.",
        "wont": "Change style or behavior — it adds facts, not skills.",
    },
    "fine-tune": {
        "changes": "the model's weights (its default behavior)",
        "reach_when": "A narrow, stable task needs consistent style/format, or you want a "
                      "smaller cheaper model to match a big one on that task.",
        "wont": "Reliably add new facts — knowledge belongs in retrieval, not weights.",
    },
}

for name, spec in LEVERS.items():
    print(f"{name:>10}  changes: {spec['changes']}")

## Encode the triage as a decision function

The chapter's triage table is really a tiny rule set. Encoding it makes the policy *executable and testable* — you can run a task through it and get a justified lever, not a hunch. The signals below are deliberately the ones the chapter calls out: volatile/attributable facts → RAG; narrow + stable + high-volume + format/behavior → fine-tune; everything else starts at the prompt.

In [ ]:
def triage(task: str) -> dict:
    """Map a task description to a recommended lever + rationale.

    Mirrors §48.1's table. Order matters: we check the 'facts' trap first
    (the most common expensive mistake), then the fine-tune bar, else prompt.
    """
    t = task.lower()

    wants_facts = any(w in t for w in [
        "cite", "citation", "current", "live", "price", "sku", "policy doc",
        "internal doc", "manual", "knowledge the model", "never seen", "inject",
    ])
    volatile = any(w in t for w in ["change", "quarterly", "live", "current", "update"])

    narrow_stable = any(w in t for w in ["narrow", "one ", "single", "stable", "schema"])
    high_volume = any(w in t for w in ["million", "volume", "cheaper", "50x", "10x", "at scale"])
    form_behavior = any(w in t for w in ["schema", "json", "format", "tone", "style", "house"])

    if wants_facts:
        return {
            "lever": "rag",
            "why": "The need is knowledge that is private/large/changing or must be cited. "
                   "Facts belong in retrieval; baking them into weights half-memorizes and "
                   "goes stale.",
        }
    if narrow_stable and (high_volume or form_behavior):
        return {
            "lever": "fine-tune",
            "why": "Narrow, stable task needing consistent form/behavior at volume. Weights "
                   "encode habit; the volume amortizes the (depreciating) training asset.",
        }
    return {
        "lever": "prompt",
        "why": "A behavior/tone/workflow change expressible as an instruction. Always try "
               "this first: reversible, cheap, no infra — and often enough.",
    }


# Smoke-test it on a couple of obvious cases.
print(triage("Adopt a warmer brand voice across all features")["lever"])
print(triage("Quote our 12,000 live SKUs and current prices")["lever"])

### 🔮 Predict

Before you run the next cell, predict the right lever for this task:

> *"Answers must cite our current internal docs, which change quarterly."*

Jot **prompt / RAG / fine-tune** and one sentence of why. The pull toward "just fine-tune on the docs" is strong — resist it and reason from *what each lever changes*. Now reveal.

In [ ]:
hard = "Answers must cite our current internal docs, which change quarterly."
verdict = triage(hard)
print("lever:", verdict["lever"])
print("why:  ", verdict["why"])

The answer is **RAG**, not fine-tune. *Current* + *cite* + *change quarterly* are three independent tells that this is a knowledge problem with an attribution and freshness requirement. Weights can't be re-attributed and can't be updated without retraining; retrieval can do both in seconds.

## Run the scenario set — check yourself against gold

`data/scenarios.json` holds tasks paired with the *correct* lever and a rationale. Run them all through `triage()` and score exact-match. This is your self-check that you internalized the table, not just memorized one example.

In [ ]:
scenarios = load_scenarios()
correct = 0
for s in scenarios:
    got = triage(s["task"])["lever"]
    ok = got == s["lever"]
    correct += ok
    mark = "ok " if ok else "MISS"
    print(f"{s['id']} [{mark}] predicted={got:<10} gold={s['lever']:<10} {s['task'][:54]}")

print(f"\ntriage accuracy: {correct}/{len(scenarios)} = {correct/len(scenarios):.0%}")
print("(Any MISS is a teaching moment: read its rationale and see which signal you'd add.)")

## ⚠️ Pitfall: fine-tuning to *inject knowledge*

The single most expensive mistake in this chapter: feeding your docs in as *training data* and expecting the model to recall them. It half-memorizes, blends facts, and hallucinates fluently in your house style — worse than vanilla RAG, and unfixable without retraining. Below is a deliberately crude mock that shows the *shape* of the failure: a "fine-tuned-on-facts" model that confidently invents, versus a RAG path that grounds and can abstain.

In [ ]:
KB = {
    "refund-policy": "Refunds are issued within 5 business days to the original card.",
    "sla": "Enterprise SLA is 99.9% monthly uptime.",
}


def answer_finetuned_on_facts(question: str) -> str:
    """Anti-pattern: facts 'trained in'. MOCK simulates lossy recall + fluent invention."""
    q = question.lower()
    # It sometimes half-remembers the right shape but corrupts the detail...
    if "refund" in q:
        return "Refunds are issued within 30 days via store credit."  # WRONG, but fluent
    if "sla" in q:
        return "Our SLA guarantees 100% uptime, always."  # WRONG, confidently
    return "Yes, that is covered under our standard policy."  # invented, no source


def answer_rag(question: str) -> str:
    """Facts in retrieval: grounded, attributable, and able to say 'I don't know'."""
    q = question.lower()
    for key, fact in KB.items():
        if key.split("-")[0] in q:
            return f"{fact} [source: {key}]"
    return "The provided context does not cover this."  # the escape hatch


for q in ["What is the refund window?", "What is the SLA?", "Do you cover water damage?"]:
    print("Q:", q)
    print("  fine-tuned-on-facts:", answer_finetuned_on_facts(q))
    print("  RAG:                ", answer_rag(q))
    print()

Read the contrast: the "trained-in facts" path is **fluent and wrong** and gives you no source to check; the RAG path is **grounded, cited, and abstains** when it doesn't know. The rule the chapter hammers: **fine-tune for form and behavior; retrieve for facts.**

## The plateau that *earns* a fine-tune

You only descend to training when prompting and RAG **demonstrably plateau on a measured eval**. Here's a tiny mocked metric across the three levers on a *narrow formatting* task: prompting and RAG lift quality but stall below the bar; only training the habit clears it. The point isn't the numbers — it's that a number, not a hunch, is what licenses the spend.

In [ ]:
def measured_score(lever: str) -> float:
    """Mocked eval score (0..1) on a narrow strict-format task. Seeded, deterministic."""
    # A task that is purely about consistent form: prompting and RAG help a bit,
    # but format adherence is a *habit* that only weight-tuning makes reliable.
    base = {"prompt": 0.71, "rag": 0.73, "fine-tune": 0.94}[lever]
    jitter = (random.random() - 0.5) * 0.02  # tiny seeded noise
    return round(min(1.0, base + jitter), 3)


BAR = 0.90  # the quality gate this task must clear to ship
print(f"target on the eval: {BAR:.0%} format-adherence\n")
for lever in ["prompt", "rag", "fine-tune"]:
    s = measured_score(lever)
    status = "clears bar" if s >= BAR else "PLATEAU below bar"
    print(f"  {lever:<10} score={s:.0%}  -> {status}")

print("\nPrompt and RAG plateau; the measured gap is the evidence that licenses training.")
print("No eval = no plateau = no justification. The eval is the prerequisite.")

## 📋 Your decision checklist

Fill this in for a real task of your own (edit the strings). It's the chapter's closing checklist, made runnable — a yes-row that's actually "no" is a reason to *not* fine-tune yet.

In [ ]:
my_task = "TODO: describe your task in one line"

checklist = {
    "prompt_and_rag_plateaued_on_a_measured_eval": False,  # set True only with numbers
    "task_is_narrow_and_stable_enough_to_amortize": False,
    "have_or_can_collect_thousands_of_curated_examples": False,
    "held_out_test_set_defined_BEFORE_training": False,
    "dataset_clean_of_pii_and_licensing_and_versioned": False,
    "serving_and_rollback_plan_exists": False,  # Ch 39: adapters on a shared base
    "owner_named_to_re_run_the_call_on_next_base_model": False,
}

print("task:", my_task)
ready = all(checklist.values())
for k, v in checklist.items():
    print(f"  [{'x' if v else ' '}] {k}")
print(f"\nVerdict: {'consider fine-tuning' if ready else 'NOT YET — stay on prompt/RAG'}")

## 🎯 Senior lens

Customization is a **portfolio decision under depreciation**. Every fine-tune is an asset that decays — the next base-model release may match your tuned model out of the box, and your adapter doesn't transfer. So capture the *durable* assets — **the curated dataset and the eval** — because those outlive any checkpoint; customize only where volume is high and the task is stable enough to amortize the work; and keep the prompt-based fallback alive so a model swap is an afternoon, not a quarter. Teams that treat data and evals as the product, and weights as a disposable artifact, win this game.

## Recap

- Three levers, three different effects: **prompt = instructions**, **RAG = knowledge**, **fine-tune = default behavior (weights)**. Reach top to bottom.
- A triage *function* makes the policy executable and self-checkable against gold scenarios.
- **Never fine-tune to inject knowledge** — it's fluent, wrong, and unattributable. Facts → retrieval; form/behavior → weights.
- Training is licensed by a **measured plateau**, not fashion. No eval, no plateau, no justification.
- The durable assets are the **dataset and the eval**; weights depreciate. Keep the prompt fallback alive.

## Exercises

1. **Break the triage.** Write a task you think `triage()` will misclassify, predict the (wrong) lever, then run it. Which *signal* would you add to `triage()` to fix it without breaking the scenarios that pass?
2. **Add a scenario.** Append one row to `data/scenarios.json` with your own gold lever + rationale. Re-run the scoring cell and confirm the accuracy line updates.
3. **Move the bar.** In the plateau cell, change `BAR` to `0.95` and to `0.70`. At which bar does fine-tuning stop being justified, and what does that tell you about choosing the gate *before* you measure?
4. **Fill the checklist for real.** Replace `my_task` and set the booleans honestly for a task at your job. If the verdict is "NOT YET," name the one row that would flip it first.

In [ ]:
# Exercise 1 — a task you think triage() gets wrong; predict, then run.


In [ ]:
# Exercise 2 — add a scenario to data/scenarios.json and re-score.


In [ ]:
# Exercise 3 — sweep BAR in {0.70, 0.95} and read the verdict.


## Next

- **Next notebook:** [`48-02-lora-peft-small-model.ipynb`](./48-02-lora-peft-small-model.ipynb) — ⚠️ optional/heavy. *If* the triage above says "fine-tune," see what LoRA actually changes (and what it doesn't), with a fully-mocked default so it still runs free.
- **The eval that gates every technique:** [`blueprints/eval-harness/`](../../../blueprints/eval-harness/) — the chapter's #keyidea is that your eval suite *is* the reward function and the rejection-sampling verifier.
- **Serving any custom model:** the `capstone/` serving + rollback plane (Ch 39) — a fine-tuned model is "an adapter or a deployment" with the same rollback discipline.